In [68]:
"""
investigate correlation in the following pairs, currently we have all predictions (x1, x2, x3, ablation, aggregate)

(1) x1+x2 vs x3
(2) x1+x3 vs x2
(3) x2+x3 vs x1

"""

'\ninvestigate correlation in the following pairs, currently we have all predictions (x1, x2, x3, ablation, aggregate)\n\n(1) x1+x2 vs x3\n(2) x1+x3 vs x2\n(3) x2+x3 vs x1\n\n'

In [1]:
import json
with open('JAADall_results.json', 'r') as f:
    results = json.load(f)

In [2]:
results.keys()

dict_keys(['x1', 'x2', 'x3', 'x1_x2', 'x1_x3', 'x2_x3', 'x1_x2_x3'])

In [3]:
def count_correlated_mistakes(preds_1, preds_2, gt):
    agreement = preds_1==preds_2
    stack_agreement_preds = []
    stack_gts = []

    for i, a in enumerate(agreement):
        if a==True:#two models agree each other
            stack_agreement_preds.append(int(preds_1[i]))
            stack_gts.append(int(gt[i]))
    aggre_num = len(np.array(stack_agreement_preds))
    mistakes_num = sum(np.array(stack_agreement_preds)!=np.array(stack_gts))
    
    return aggre_num, mistakes_num

def count_mistake_union(preds_1, preds_2, gt):
    mistake_1 = []#agreement
    mistake_2 = []#agreed mistakes
    
    for i, v in enumerate(gt):
        if int(gt[i])!= int(preds_1[i]):
            mistake_1.append(i)
        
        if int(gt[i])!= int(preds_2[i]):
            mistake_2.append(i)
    return len(set(list(set(mistake_1)) + list(set(mistake_2))))


def count_indivisual_mistakes(preds, gt):
    return sum(preds!=gt)

def get_conf_coef(probs_1, probs_2):
    return np.corrcoef(np.squeeze(probs_1), np.squeeze(probs_2))#x2_x3

# x1

In [4]:
def flatten_transformNP(model_name):
    preds = np.array(results[model_name]['preds'])
    src_li = np.array(results[model_name]['test_gts'])
    if not preds.shape == src_li.shape:
        mod_li = list(itertools.chain(*src_li))
        print(count_indivisual_mistakes(preds, mod_li))
    else:
        print(count_indivisual_mistakes(preds,src_li))

In [6]:
import numpy as np
flatten_transformNP(model_name='x1')

91088


# x2

In [7]:
flatten_transformNP(model_name='x2')

170957


# x3

In [8]:
flatten_transformNP(model_name='x3')

150931


# x1 + x2

In [11]:
import itertools
base = 'x1'
add = 'x2'
preds_1 = np.array(results[base]['preds'])
preds_2 = np.array(results[add]['preds'])
gt = np.array(results[base]['test_gts'])
if not gt.shape==preds_2.shape:
    gt = list(itertools.chain(*gt))
print('agreed prediction and agreed mistakes')
print(count_correlated_mistakes(preds_1, preds_2, gt))
print('actual mistakes')
merged = base+'_'+add
preds = np.array(results[merged]['preds'])
print(count_indivisual_mistakes(preds, gt))
print('mistake union')
print(count_mistake_union(preds_1, preds_2, gt))
print('correlation coefficient')
probs_1 = np.array(results[base]['probs'])
probs_2 = np.array(results[add]['probs'])
print(get_conf_coef(probs_1, probs_2))
print('total gt')
print(len(gt))

agreed prediction and agreed mistakes
(278416, 34478)
actual mistakes
84650
mistake union
227567
correlation coefficient
[[1.         0.06247631]
 [0.06247631 1.        ]]
total gt
471505


# x1+x3

In [12]:
base = 'x1'
add = 'x3'
def get_all_score(base, add):
    preds_1 = np.array(results[base]['preds'])
    preds_2 = np.array(results[add]['preds'])
    gt = np.array(results[base]['test_gts'])
    if not gt.shape == preds_2.shape:
        gt = list(itertools.chain(*gt))
    print('agreed prediction and agreed mistakes')
    print(count_correlated_mistakes(preds_1, preds_2, gt))
    print('actual mistakes')
    merged = base+'_'+add
    if len(merged.split('_'))==3:
        merged = 'x1_x2_x3'
    preds = np.array(results[merged]['preds'])
    print(count_indivisual_mistakes(preds, gt))
    print('mistake union')
    print(count_mistake_union(preds_1, preds_2, gt))
    print('correlation coefficient')
    probs_1 = np.array(results[base]['probs'])
    probs_2 = np.array(results[add]['probs'])
    print(get_conf_coef(probs_1, probs_2))
    print('base model mistakes')
    print(count_indivisual_mistakes(preds_1, gt))
    print('additional model mistakes')
    print(count_indivisual_mistakes(preds_2, gt))
    print('total gt')
    print(len(gt))

In [13]:
get_all_score(base, add)

agreed prediction and agreed mistakes
(319210, 44862)
actual mistakes
81749
mistake union
197157
correlation coefficient
[[1.         0.28396919]
 [0.28396919 1.        ]]
base model mistakes
91088
additional model mistakes
150931
total gt
471505


# x2 + x3

In [14]:
base = 'x2'
add = 'x3'
get_all_score(base, add)

agreed prediction and agreed mistakes
(259917, 55150)
actual mistakes
118766
mistake union
266738
correlation coefficient
[[ 1.         -0.01457234]
 [-0.01457234  1.        ]]
base model mistakes
170957
additional model mistakes
150931
total gt
471505


# (1) x1+x2 vs x3

In [16]:
base = 'x1_x2'
add = 'x3'
get_all_score(base, add)

agreed prediction and agreed mistakes
(315232, 39654)
actual mistakes
56958
mistake union
195927
correlation coefficient
[[1.         0.21802743]
 [0.21802743 1.        ]]
base model mistakes
84650
additional model mistakes
150931
total gt
471505


# (2) x1+x3 vs x2

In [17]:
base = 'x1_x3'
add = 'x2'
get_all_score(base, add)

agreed prediction and agreed mistakes
(279441, 30321)
actual mistakes
56958
mistake union
222385
correlation coefficient
[[1.         0.05567891]
 [0.05567891 1.        ]]
base model mistakes
81749
additional model mistakes
170957
total gt
471505


# (3) x2+x3 vs x1

In [18]:
base = 'x2_x3'
add = 'x1'
get_all_score(base, add)

agreed prediction and agreed mistakes
(334135, 36242)
actual mistakes
56958
mistake union
173612
correlation coefficient
[[1.         0.28323482]
 [0.28323482 1.        ]]
base model mistakes
118766
additional model mistakes
91088
total gt
471505


# x1_x2_x3 mistakes

In [66]:
x1_x2_x3_preds = np.array(results['x1_x2_x3']['preds'])
count_indivisual_mistakes(x1_x2_x3_preds, gt)

31668